# Artisan: Pipelines with Provenance

Define steps. Wire outputs to inputs. Run.

The framework handles caching, lineage tracking, and storage.

---

## The Problem

You run a pipeline. It produces results. A week later someone asks:

> *Which input data and which parameters produced this specific output?*

You dig through logs, filenames, and notebooks. It takes an hour.

Artisan makes that question **instant** — every artifact is tracked with
full provenance from the moment it's created.

---

## Setup

In [ ]:
from artisan.operations.curator import Filter, IngestData, Merge
from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager
from artisan.utils import find_project_root, tutorial_setup
from artisan.visualization import (
    build_macro_graph,
    inspect_metrics,
    inspect_pipeline,
)

PROJECT_ROOT = find_project_root()
SOURCE_FILES = sorted((PROJECT_ROOT / "tests" / "fixtures" / "csv").glob("*.csv"))[:2]

env = tutorial_setup("demo")
DELTA_ROOT = env.delta_root

---

## Building a Pipeline

A pipeline is a sequence of **steps**. Each step runs an **operation** —
a unit of computation with declared inputs and outputs.

You wire steps together with **output references**: `output(step_name, role)`.

In [ ]:
pipeline = PipelineManager.create(
    name="demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

### Ingest external data

Bring CSV files into the pipeline's tracking system.

In [ ]:
pipeline.run(
    operation=IngestData,
    name="ingest",
    inputs=[str(f) for f in SOURCE_FILES],
)

### Generate synthetic data

A second data source — no inputs, just parameters.

In [ ]:
pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 2, "seed": 44},
)

### Merge the two streams

Combine ingest and generate into a single stream.
This is where **output references** connect the steps.

In [ ]:
pipeline.run(
    operation=Merge,
    name="merge",
    inputs={
        "branch_a": output("ingest", "data"),
        "branch_b": output("generate", "datasets"),
    },
)

### Transform

Scale every dataset. The framework processes each artifact independently.

In [ ]:
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("merge", "merged")},
    params={"scale_factor": 0.5, "variants": 1, "seed": 100},
)

### Compute metrics

Score each dataset — the metric artifacts are stored alongside the data.

In [ ]:
pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
)

### Filter by score

Keep only datasets where the median score exceeds a threshold.

Filter **auto-discovers** the relevant metrics by walking provenance
forward from the passthrough artifacts — no explicit link to the
metrics step needed.

In [ ]:
pipeline.run(
    operation=Filter,
    name="filter",
    inputs={"passthrough": output("transform", "dataset")},
    params={
        "criteria": [
            {"metric": "distribution.median", "operator": "gt", "value": 0.15},
        ]
    },
)

In [ ]:
result = pipeline.finalize()
print(f"Pipeline complete: {result['total_steps']} steps, success={result['overall_success']}")

---

## The Payoff

The pipeline ran. Now let's answer that question:
*what happened, and where did it come from?*

### Pipeline summary

One row per step — what ran, what it produced, how long it took.

In [ ]:
inspect_pipeline(DELTA_ROOT)

### Queryable metrics

All metric values are stored in Delta Lake — query them as DataFrames.

In [ ]:
inspect_metrics(DELTA_ROOT, step_number=4).sort("distribution.median", descending=True)

### Provenance graph

The full pipeline topology — every step and data flow — rendered automatically.

In [ ]:
build_macro_graph(DELTA_ROOT)

---

## Key Ideas

- **Operations** are pure computation — inputs in, artifacts out
- **Provenance** is captured automatically — zero manual wiring
- **Content addressing** gives you deterministic caching for free
- **Everything is queryable** — artifacts, metrics, and lineage live in Delta Lake
- **Same code scales** from laptop to HPC cluster